# Fase 3 — Exportación del modelo a ONNX

Convertimos `model/best.pt` a formato **ONNX** para inferencia en la app móvil con `onnxruntime-web`.

**¿Por qué ONNX y no TFLite?**
- La app es Ionic + React + Capacitor → corre dentro de un WebView. `onnxruntime-web` se integra trivialmente con cualquier proyecto JS/TS, no requiere plugins nativos.
- TFLite requiere puentes nativos (no aptos para Ionic clásico).

**Configuración:**
- `imgsz=640` (igual al entrenamiento).
- `opset=12` (compatible con onnxruntime-web).
- `simplify=True` (optimiza el grafo, modelo más pequeño y rápido).
- `dynamic=False` (batch fijo = 1, mejor para inferencia en tiempo real).

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
from pathlib import Path
import shutil
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "model"
BEST_PT = MODEL_DIR / "best.pt"
assert BEST_PT.exists(), f"No se encontró {BEST_PT}"

model = YOLO(str(BEST_PT))
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False,
    half=False,
    device="cpu",
)
print(f"Exportado: {onnx_path}")

In [ ]:
# Copiar el ONNX a la carpeta pública de la app y a model/ para versionado.
src = Path(onnx_path)
model_dst = MODEL_DIR / "best.onnx"
app_models = ROOT / "app" / "public" / "models"
app_models.mkdir(parents=True, exist_ok=True)
app_dst = app_models / "coin-detector.onnx"

shutil.copy2(src, model_dst)
shutil.copy2(src, app_dst)
for p in [model_dst, app_dst]:
    print(f"  {p} ({p.stat().st_size / 1024 / 1024:.2f} MB)")

## Verificación: cargar el ONNX y comprobar input/output shapes

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(str(model_dst), providers=["CPUExecutionProvider"])
for inp in session.get_inputs():
    print(f"INPUT  {inp.name}: shape={inp.shape}, type={inp.type}")
for out in session.get_outputs():
    print(f"OUTPUT {out.name}: shape={out.shape}, type={out.type}")

# Inferencia dummy para verificar funcionamiento
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
out = session.run(None, {session.get_inputs()[0].name: dummy})
print(f"\nOutput shape: {out[0].shape}")  # (1, 4+nc, 8400) — YOLOv8 detect
print(f"Output dtype: {out[0].dtype}")

## Notas para la integración móvil

El modelo ONNX exportado tiene:
- **Input:** `images` shape `(1, 3, 640, 640)` float32, valores normalizados a `[0, 1]`, formato CHW (channels, height, width), RGB.
- **Output:** `output0` shape `(1, 9, 8400)` donde:
  - 9 = 4 (bbox: `xc, yc, w, h` en píxeles del frame 640×640) + 5 (probabilidades por clase: `[100, 1000, 200, 50, 500]`).
  - 8400 = anchors totales (P3+P4+P5 a stride 8/16/32).

El post-procesado en JS debe:
1. Transponer a `(8400, 9)`.
2. Filtrar anchors con `max(class_score) >= conf_thresh`.
3. Aplicar **Non-Maximum Suppression (NMS)** con `iou_thresh=0.45`.
4. Mapear coordenadas de 640×640 a las del frame original (considerando el resize stretch o letterbox usado en preprocesamiento).